# InsureIQ — Insurance Claims Analytics & Risk Intelligence
**End-to-end Google Colab notebook:** data loading, cleaning, EDA, insurance metrics, visualizations, anomaly detection, Random Forest regression, evaluation, explainability and business insights.

**Target:** `Claim_Amount` (continuous numerical outcome).  
**Important:** This synthetic dataset is for educational analysis. Model estimates are not claim settlement decisions. `Claim_Status` and `Claim_Count` are excluded from the primary model because they may reveal information that would not be available at prediction time. `Claim_Count` is also closely tied to claim occurrence.

## 1. Import libraries

In [ ]:
import warnings, os, io, json, math
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(context="notebook", style="whitegrid")

## 2. Upload and load the dataset

In [ ]:
# Option A: Upload the CSV from your computer (recommended in Colab)
from google.colab import files
uploaded = files.upload()
csv_name = next((name for name in uploaded if name.lower().endswith(".csv")), None)
if csv_name is None:
    raise ValueError("Please upload a CSV file.")
df_raw = pd.read_csv(io.BytesIO(uploaded[csv_name]))

# Option B: If the file is already in /content, comment out Option A and use:
# df_raw = pd.read_csv("/content/insureiq_insurance_claims_10000.csv")

print("Dataset shape:", df_raw.shape)
display(df_raw.head())
display(df_raw.sample(min(5, len(df_raw)), random_state=RANDOM_STATE))

## 3. Initial data audit

In [ ]:
print("Shape:", df_raw.shape)
print("\nData types:")
display(df_raw.dtypes.rename("dtype").to_frame())
print("\nMissing values:")
display(df_raw.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("\nDuplicate rows:", df_raw.duplicated().sum())
if "Policy_ID" in df_raw:
    print("Duplicate Policy_IDs:", df_raw["Policy_ID"].duplicated().sum())
print("\nNumeric summary:")
display(df_raw.describe().T)
print("\nCategorical summary:")
display(df_raw.describe(include="object").T)

## 4. Data cleaning and validation
The original data is retained in `df_raw`. Cleaning is performed on a copy. Negative monetary values and invalid ages are treated as invalid; extreme but plausible claims are flagged, not automatically deleted.

In [ ]:
df = df_raw.copy()
required = ["Policy_ID", "Region", "Premium", "Claim_Amount",
            "Policy_Type", "Vehicle_Property_Type", "Customer_Age"]
missing_required = [c for c in required if c not in df.columns]
if missing_required:
    raise ValueError(f"Required columns missing: {missing_required}")

# Standardize text columns
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].astype("string").str.strip()
    df[col] = df[col].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})

# Convert known numeric columns safely
numeric_cols = ["Premium", "Claim_Amount", "Customer_Age", "Policy_Duration",
                "Coverage_Amount", "Deductible", "Claim_Count", "Policy_Year",
                "Vehicle_Age", "Property_Age", "Previous_Claims"]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

if "Policy_Start_Date" in df.columns:
    df["Policy_Start_Date"] = pd.to_datetime(df["Policy_Start_Date"], errors="coerce")

# Remove exact duplicate rows, then retain the first occurrence of duplicate IDs
before = len(df)
df = df.drop_duplicates()
exact_duplicates_removed = before - len(df)
duplicate_ids_removed = 0
if df["Policy_ID"].notna().any():
    before = len(df)
    df = df.drop_duplicates(subset=["Policy_ID"], keep="first")
    duplicate_ids_removed = before - len(df)

# Remove rows with invalid core monetary/age values; do not delete valid high claims
invalid_mask = (
    df["Premium"].isna() | (df["Premium"] < 0) |
    df["Claim_Amount"].isna() | (df["Claim_Amount"] < 0) |
    df["Customer_Age"].isna() | (~df["Customer_Age"].between(18, 100))
)
invalid_core_removed = int(invalid_mask.sum())
df = df.loc[~invalid_mask].copy()

# Validate optional numeric fields: mark impossible values as missing for pipeline imputation
for col in ["Policy_Duration", "Coverage_Amount", "Deductible",
            "Claim_Count", "Vehicle_Age", "Property_Age", "Previous_Claims"]:
    if col in df.columns:
        df.loc[df[col] < 0, col] = np.nan
if "Coverage_Amount" in df.columns:
    df.loc[df["Coverage_Amount"] <= 0, "Coverage_Amount"] = np.nan
if "Deductible" in df.columns and "Coverage_Amount" in df.columns:
    df.loc[df["Deductible"] > df["Coverage_Amount"], "Deductible"] = np.nan

# Normalize category spelling/capitalization (trim only; retain original category labels)
for col in ["Region", "Policy_Type", "Vehicle_Property_Type", "Customer_Type",
            "Risk_Profile", "Claim_Status", "Location_Type"]:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

cleaning_summary = pd.DataFrame({
    "Metric": ["Original rows", "Exact duplicates removed", "Duplicate IDs removed",
               "Invalid core rows removed", "Final rows", "Final columns",
               "Remaining missing cells"],
    "Value": [len(df_raw), exact_duplicates_removed, duplicate_ids_removed,
              invalid_core_removed, len(df), df.shape[1], int(df.isna().sum().sum())]
})
display(cleaning_summary)
display(df.head())

## 5. Data quality and outlier flags

In [ ]:
# IQR flags are exploratory; flagged observations are not automatically removed.
outlier_summary = []
for col in ["Premium", "Claim_Amount", "Coverage_Amount", "Customer_Age"]:
    if col in df.columns:
        x = df[col].dropna()
        q1, q3 = x.quantile(.25), x.quantile(.75)
        iqr = q3 - q1
        low, high = q1 - 1.5*iqr, q3 + 1.5*iqr
        outlier_summary.append([col, int(((x < low) | (x > high)).sum()), low, high])
display(pd.DataFrame(outlier_summary, columns=["Feature","IQR flagged count","Lower fence","Upper fence"]))
print("Missing values after cleaning:")
display(df.isna().sum().to_frame("missing"))

## 6. Feature engineering and insurance metrics

In [ ]:
# Define claim occurrence from positive incurred claim amount.
df["Has_Claim"] = (df["Claim_Amount"] > 0).astype(int)

def insurance_summary(data):
    n_policies = len(data)
    n_claiming = int((data["Claim_Amount"] > 0).sum())
    total_claims = data["Claim_Amount"].sum()
    total_premium = data["Premium"].sum()
    return pd.Series({
        "Policies": n_policies,
        "Policies with claims": n_claiming,
        "Claim frequency (%)": 100*n_claiming/n_policies if n_policies else np.nan,
        "Total claim amount": total_claims,
        "Claim severity (per claiming policy)": (
            total_claims/n_claiming if n_claiming else np.nan),
        "Average claim amount per policy": total_claims/n_policies if n_policies else np.nan,
        "Total premium": total_premium,
        "Simplified claims-to-premium ratio (%)": (
            100*total_claims/total_premium if total_premium else np.nan)
    })

display(insurance_summary(df).to_frame("Value"))
print("Note: Claims-to-premium ratio is simplified; earned premium is not supplied.")

## 7. Exploratory data analysis — descriptive statistics

In [ ]:
numeric_present = [c for c in numeric_cols if c in df.columns]
display(df[numeric_present].describe(percentiles=[.25,.5,.75,.9,.95,.99]).T)
display(df[numeric_present].agg(["mean","median","std","var","skew","kurt"]).T)

## 8. Visualizations — distributions and relationships

In [ ]:
# 1. Claim amount distribution (zeros included)
plt.figure(figsize=(10,5))
sns.histplot(df["Claim_Amount"], bins=60, kde=False)
plt.title("Claim Amount Distribution (including zero-claim policies)")
plt.xlabel("Claim amount"); plt.ylabel("Policy count"); plt.tight_layout(); plt.show()

# 2. Positive claim severity distribution
positive_claims = df.loc[df["Claim_Amount"] > 0, "Claim_Amount"]
plt.figure(figsize=(10,5))
sns.histplot(positive_claims, bins=50, kde=True)
plt.title("Claim Severity Distribution (positive claims only)")
plt.xlabel("Claim amount"); plt.tight_layout(); plt.show()

# 3. Premium distribution
plt.figure(figsize=(10,5))
sns.histplot(df["Premium"], bins=50, kde=True)
plt.title("Premium Distribution"); plt.xlabel("Premium"); plt.tight_layout(); plt.show()

# 4. Boxplot of claim amount by policy type
plt.figure(figsize=(12,6))
sns.boxplot(data=df, x="Policy_Type", y="Claim_Amount", showfliers=False)
plt.title("Claim Amount by Policy Type (fliers hidden for readability)")
plt.xticks(rotation=25); plt.tight_layout(); plt.show()

# 5. Correlation heatmap
corr_cols = [c for c in numeric_present if df[c].nunique(dropna=True)>1]
plt.figure(figsize=(11,8))
sns.heatmap(df[corr_cols].corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Numeric Feature Correlation"); plt.tight_layout(); plt.show()

# 6. Premium vs claim amount
plt.figure(figsize=(10,6))
sns.scatterplot(data=df.sample(min(3000,len(df)), random_state=RANDOM_STATE),
                x="Premium", y="Claim_Amount", hue="Policy_Type", alpha=.6)
plt.title("Premium vs Claim Amount (sampled for readability)")
plt.tight_layout(); plt.show()

## 9. Claim frequency and severity analysis

In [ ]:
def grouped_claim_metrics(data, by):
    out = data.groupby(by, dropna=False).agg(
        Policies=("Policy_ID","count"),
        Claiming_Policies=("Has_Claim","sum"),
        Total_Claims=("Claim_Amount","sum"),
        Total_Premium=("Premium","sum")
    )
    out["Claim_Frequency_%"] = 100*out["Claiming_Policies"]/out["Policies"]
    out["Claim_Severity"] = out["Total_Claims"]/out["Claiming_Policies"].replace(0,np.nan)
    out["Avg_Claim_Per_Policy"] = out["Total_Claims"]/out["Policies"]
    out["Claims_to_Premium_%"] = 100*out["Total_Claims"]/out["Total_Premium"].replace(0,np.nan)
    return out.sort_values("Claim_Frequency_%", ascending=False)

region_metrics = grouped_claim_metrics(df, "Region")
policy_metrics = grouped_claim_metrics(df, "Policy_Type")
asset_metrics = grouped_claim_metrics(df, "Vehicle_Property_Type")
display(region_metrics)
display(policy_metrics)
display(asset_metrics)

# 7. Regional claim frequency
ax = region_metrics["Claim_Frequency_%"].sort_values().plot(kind="barh", figsize=(9,5))
ax.set_title("Claim Frequency by Region"); ax.set_xlabel("Claim frequency (%)"); plt.tight_layout(); plt.show()

# 8. Regional severity
ax = region_metrics["Claim_Severity"].sort_values().plot(kind="barh", figsize=(9,5))
ax.set_title("Claim Severity by Region"); ax.set_xlabel("Average claim per claiming policy"); plt.tight_layout(); plt.show()

# 9. Regional claims-to-premium ratio
ax = region_metrics["Claims_to_Premium_%"].sort_values().plot(kind="barh", figsize=(9,5))
ax.set_title("Simplified Claims-to-Premium Ratio by Region"); ax.set_xlabel("Ratio (%)"); plt.tight_layout(); plt.show()

# 10. Policy-type frequency and severity
fig, axes = plt.subplots(1,2,figsize=(13,5))
policy_metrics["Claim_Frequency_%"].plot(kind="bar",ax=axes[0],title="Frequency by Policy Type")
policy_metrics["Claim_Severity"].plot(kind="bar",ax=axes[1],title="Severity by Policy Type")
for ax in axes: ax.tick_params(axis="x",rotation=30)
plt.tight_layout(); plt.show()

## 10. Customer and asset segment analysis

In [ ]:
df["Age_Group"] = pd.cut(df["Customer_Age"], bins=[17,24,34,44,54,64,100],
                         labels=["Under 25","25–34","35–44","45–54","55–64","65+"])
age_metrics = grouped_claim_metrics(df, "Age_Group")
display(age_metrics)

# 11. Age group claim frequency
age_metrics["Claim_Frequency_%"].plot(kind="bar", figsize=(9,5), title="Claim Frequency by Customer Age Group")
plt.ylabel("Claim frequency (%)"); plt.tight_layout(); plt.show()

# 12. Asset type severity
asset_metrics["Claim_Severity"].sort_values().plot(kind="barh", figsize=(10,6), title="Claim Severity by Asset Type")
plt.xlabel("Average claim per claiming policy"); plt.tight_layout(); plt.show()

# 13. Risk profile comparison
risk_metrics = grouped_claim_metrics(df, "Risk_Profile")
display(risk_metrics)
risk_metrics[["Claim_Frequency_%","Claims_to_Premium_%"]].plot(kind="bar", figsize=(9,5))
plt.title("Frequency and Claims-to-Premium Ratio by Risk Profile")
plt.ylabel("Percent"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

# 14. Region-policy type heatmap: claim frequency
pivot = df.pivot_table(index="Region", columns="Policy_Type", values="Has_Claim", aggfunc="mean")*100
plt.figure(figsize=(10,6))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="YlOrRd")
plt.title("Claim Frequency Heatmap: Region × Policy Type")
plt.tight_layout(); plt.show()

## 11. Adjusted regional comparison (regression standardization)
This is an observational, model-based adjustment for the available policy characteristics. It is not a causal estimate. The adjustment model is fitted to the full cleaned dataset for descriptive standardization only; the predictive Random Forest below uses a separate held-out test set.

In [ ]:
# Estimate expected claim amounts using non-geographic policy/customer characteristics.
# Region is intentionally excluded from this baseline adjustment model.
adjust_features = [c for c in [
    "Policy_Type","Vehicle_Property_Type","Customer_Age","Coverage_Amount",
    "Deductible","Customer_Type","Risk_Profile","Location_Type",
    "Vehicle_Age","Property_Age","Previous_Claims","Policy_Duration"
] if c in df.columns]

if adjust_features:
    adj_num = [c for c in adjust_features if pd.api.types.is_numeric_dtype(df[c])]
    adj_cat = [c for c in adjust_features if c not in adj_num]
    adj_pre = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), adj_num),
        ("cat", Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),
                          ("onehot",OneHotEncoder(handle_unknown="ignore"))]), adj_cat)
    ])
    from sklearn.ensemble import RandomForestRegressor as RFR
    adj_model = Pipeline([("preprocess",adj_pre),
                          ("model",RFR(n_estimators=200,min_samples_leaf=5,
                                       random_state=RANDOM_STATE,n_jobs=-1))])
    adj_model.fit(df[adjust_features], df["Claim_Amount"])
    df["Expected_Claim_Adjustment"] = adj_model.predict(df[adjust_features])
    df["Expected_Claim_Adjustment"] = df["Expected_Claim_Adjustment"].clip(lower=0)
    adjusted = df.groupby("Region").agg(
        Policies=("Policy_ID","count"),
        Observed_Avg_Claim=("Claim_Amount","mean"),
        Expected_Avg_Claim=("Expected_Claim_Adjustment","mean")
    )
    adjusted["Observed_to_Expected"] = (
        adjusted["Observed_Avg_Claim"] /
        adjusted["Expected_Avg_Claim"].replace(0,np.nan)
    )
    adjusted["Adjusted_Difference"] = (
        adjusted["Observed_Avg_Claim"]-adjusted["Expected_Avg_Claim"]
    )
    display(adjusted.sort_values("Observed_to_Expected",ascending=False))
    adjusted[["Observed_Avg_Claim","Expected_Avg_Claim"]].plot(
        kind="bar",figsize=(11,5),title="Observed vs Expected Average Claim by Region")
    plt.ylabel("Claim amount"); plt.xticks(rotation=25); plt.tight_layout(); plt.show()
else:
    print("Insufficient fields for adjusted regional comparison.")

## 12. Anomaly detection

In [ ]:
# IQR flag for large claim amounts, plus Isolation Forest for multivariate unusual records.
claim_values = df["Claim_Amount"]
q1, q3 = claim_values.quantile(.25), claim_values.quantile(.75)
iqr = q3-q1
upper_fence = q3+1.5*iqr
df["IQR_Claim_Flag"] = df["Claim_Amount"] > upper_fence

anomaly_features = [c for c in [
    "Premium","Claim_Amount","Coverage_Amount","Deductible","Customer_Age",
    "Previous_Claims","Vehicle_Age","Property_Age"
] if c in df.columns]
anom_data = df[anomaly_features].copy()
anom_data = anom_data.replace([np.inf,-np.inf],np.nan)
anom_data = anom_data.fillna(anom_data.median(numeric_only=True)).fillna(0)

iso = IsolationForest(contamination=0.02, random_state=RANDOM_STATE)
df["IsolationForest_Flag"] = iso.fit_predict(anom_data) == -1
df["Anomaly_Score"] = -iso.decision_function(anom_data)
df["Anomaly_Flag"] = df["IQR_Claim_Flag"] | df["IsolationForest_Flag"]

anomalies = df.loc[df["Anomaly_Flag"], [
    "Policy_ID","Region","Policy_Type","Claim_Amount","Premium",
    "Coverage_Amount","IQR_Claim_Flag","IsolationForest_Flag","Anomaly_Score"
]].sort_values("Anomaly_Score",ascending=False)
print("Flagged observations:", len(anomalies))
display(anomalies.head(20))

# 15. Anomaly score distribution
plt.figure(figsize=(9,5))
sns.histplot(df["Anomaly_Score"], bins=50, kde=True)
plt.title("Isolation Forest Anomaly Score Distribution")
plt.tight_layout(); plt.show()

# 16. Claims flagged by region
df.groupby("Region")["Anomaly_Flag"].mean().mul(100).sort_values().plot(
    kind="barh",figsize=(9,5),title="Percentage of Records Flagged by Region")
plt.xlabel("Flagged records (%)"); plt.tight_layout(); plt.show()
print("Anomaly flags indicate unusual records for review, not fraud.")

## 13. Prepare data for Random Forest regression
The main model predicts the policy-level incurred `Claim_Amount`, including zero claims. Do not use claim-derived fields such as `Claim_Status` or `Claim_Count` as predictors.

In [ ]:
target = "Claim_Amount"
excluded = {
    "Policy_ID", "Claim_Amount", "Claim_Status", "Claim_Count",
    "Has_Claim", "Age_Group", "IQR_Claim_Flag", "IsolationForest_Flag",
    "Anomaly_Score", "Anomaly_Flag", "Expected_Claim_Adjustment"
}
feature_cols = [c for c in df.columns if c not in excluded]

# Exclude dates as raw strings; derive policy year if available.
if "Policy_Start_Date" in feature_cols:
    feature_cols.remove("Policy_Start_Date")
if "Policy_Year" in df.columns and "Policy_Year" not in feature_cols:
    feature_cols.append("Policy_Year")

X = df[feature_cols].copy()
y = df[target].astype(float).copy()

# Remove all-empty features
feature_cols = [c for c in feature_cols if not X[c].isna().all()]
X = X[feature_cols]

numeric_features = X.select_dtypes(include=["number","bool"]).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print("Features:", feature_cols)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train/test overlap:", len(set(X_train.index).intersection(set(X_test.index))))

## 14. Train Random Forest and tune hyperparameters

In [ ]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1))
])

param_distributions = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": [1.0, "sqrt", "log2"]
}
cv = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_distributions,
    n_iter=12, scoring="neg_mean_absolute_error",
    cv=cv, random_state=RANDOM_STATE, n_jobs=-1,
    verbose=1, refit=True
)
search.fit(X_train, y_train)
best_model = search.best_estimator_
print("Best CV MAE:", -search.best_score_)
print("Best parameters:", search.best_params_)

## 15. Model evaluation — regression metrics
There is no conventional classification accuracy for this regression target. MAE, RMSE and R² are the principal metrics. MAPE is reported only for positive actual claims because zero values make it undefined.

In [ ]:
y_pred = np.maximum(0, best_model.predict(X_test))
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
bias = float(np.mean(y_pred-y_test))
positive_mask = y_test > 0
positive_mae = mean_absolute_error(y_test[positive_mask], y_pred[positive_mask]) if positive_mask.any() else np.nan
zero_mask = y_test == 0
zero_mae = mean_absolute_error(y_test[zero_mask], y_pred[zero_mask]) if zero_mask.any() else np.nan
mape = (np.mean(np.abs((y_test[positive_mask]-y_pred[positive_mask])/y_test[positive_mask]))*100
        if positive_mask.any() else np.nan)

metrics = pd.DataFrame({
    "Metric": ["MAE","RMSE","R²","Mean prediction bias (predicted - actual)",
               "MAE: positive-claim policies","MAE: zero-claim policies",
               "MAPE: positive claims only (%)"],
    "Value": [mae,rmse,r2,bias,positive_mae,zero_mae,mape]
})
display(metrics)

## 16. Baseline comparison

In [ ]:
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
baseline_pred = np.maximum(0, baseline.predict(X_test))
comparison = pd.DataFrame({
    "Model":["Training mean baseline","Random Forest"],
    "MAE":[mean_absolute_error(y_test,baseline_pred),mae],
    "RMSE":[np.sqrt(mean_squared_error(y_test,baseline_pred)),rmse],
    "R2":[r2_score(y_test,baseline_pred),r2]
})
display(comparison)
comparison.set_index("Model")[["MAE","RMSE"]].plot(kind="bar",figsize=(9,5),title="Baseline vs Random Forest Error")
plt.ylabel("Error (lower is better)"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 17. Actual vs predicted, residuals and evaluation curves

In [ ]:
results = pd.DataFrame({"Actual":y_test.values,"Predicted":y_pred})
results["Residual"] = results["Actual"]-results["Predicted"]
results["Absolute_Error"] = results["Residual"].abs()

# 17. Actual vs predicted
plt.figure(figsize=(7,7))
sns.scatterplot(data=results.sample(min(3000,len(results)),random_state=RANDOM_STATE),
                x="Actual",y="Predicted",alpha=.5)
lims=[0,max(results["Actual"].max(),results["Predicted"].max())]
plt.plot(lims,lims,"--",label="Ideal prediction")
plt.xlim(lims); plt.ylim(lims); plt.xlabel("Actual claim amount"); plt.ylabel("Predicted claim amount")
plt.title("Actual vs Predicted Claim Amount"); plt.legend(); plt.tight_layout(); plt.show()

# 18. Residual distribution
plt.figure(figsize=(9,5))
sns.histplot(results["Residual"],bins=50,kde=True)
plt.axvline(0,linestyle="--")
plt.title("Residual Distribution (Actual − Predicted)")
plt.tight_layout(); plt.show()

# 19. Residuals vs predictions
plt.figure(figsize=(9,5))
sns.scatterplot(data=results.sample(min(3000,len(results)),random_state=RANDOM_STATE),
                x="Predicted",y="Residual",alpha=.5)
plt.axhline(0,linestyle="--")
plt.title("Residuals vs Predicted Values")
plt.xlabel("Predicted claim amount"); plt.ylabel("Residual")
plt.tight_layout(); plt.show()

# 20. Absolute error distribution
plt.figure(figsize=(9,5))
sns.histplot(results["Absolute_Error"],bins=50,kde=True)
plt.title("Absolute Prediction Error Distribution")
plt.xlabel("Absolute error"); plt.tight_layout(); plt.show()

# 21. Error by actual-claim status
results["Claim_Group"] = np.where(results["Actual"]>0,"Positive claim","Zero claim")
sns.boxplot(data=results,x="Claim_Group",y="Absolute_Error",showfliers=False)
plt.title("Absolute Error by Claim Group"); plt.tight_layout(); plt.show()

## 18. Model performance by region and policy type

In [ ]:
eval_df = df.loc[X_test.index, ["Region","Policy_Type"]].copy()
eval_df["Actual"] = y_test
eval_df["Predicted"] = y_pred
eval_df["Absolute_Error"] = np.abs(eval_df["Actual"]-eval_df["Predicted"])

region_eval = eval_df.groupby("Region").agg(
    Policies=("Actual","size"),
    MAE=("Absolute_Error","mean"),
    Actual_Mean=("Actual","mean"),
    Predicted_Mean=("Predicted","mean")
)
policy_eval = eval_df.groupby("Policy_Type").agg(
    Policies=("Actual","size"),
    MAE=("Absolute_Error","mean"),
    Actual_Mean=("Actual","mean"),
    Predicted_Mean=("Predicted","mean")
)
display(region_eval)
display(policy_eval)
region_eval["MAE"].sort_values().plot(kind="barh",figsize=(9,5),title="Test MAE by Region")
plt.xlabel("MAE"); plt.tight_layout(); plt.show()
policy_eval["MAE"].sort_values().plot(kind="barh",figsize=(9,5),title="Test MAE by Policy Type")
plt.xlabel("MAE"); plt.tight_layout(); plt.show()

## 19. Feature importance and permutation importance

In [ ]:
# Impurity-based importance
rf = best_model.named_steps["model"]
feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
imp = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)
display(imp.head(20).to_frame("Random Forest importance"))
imp.head(20).sort_values().plot(kind="barh",figsize=(10,7),title="Top 20 Impurity-Based Feature Importances")
plt.xlabel("Importance"); plt.tight_layout(); plt.show()

# Permutation importance on held-out test set; scoring is negative MAE.
perm = permutation_importance(
    best_model, X_test, y_test, n_repeats=5,
    random_state=RANDOM_STATE, scoring="neg_mean_absolute_error", n_jobs=-1
)
perm_imp = pd.Series(perm.importances_mean,index=X_test.columns).sort_values(ascending=False)
display(perm_imp.to_frame("Permutation importance (MAE increase)"))
perm_imp.sort_values().plot(kind="barh",figsize=(9,6),title="Permutation Importance on Test Data")
plt.xlabel("Increase in MAE when shuffled"); plt.tight_layout(); plt.show()

## 20. Generate 5–7 data-driven business insights

In [ ]:
insights = []
overall = insurance_summary(df)
insights.append({
    "Insight":"Overall claims experience",
    "Evidence":f'{overall["Policies with claims"]:,} of {overall["Policies"]:,} policies had a positive claim; frequency {overall["Claim frequency (%)"]:.2f}%.',
    "Implication":"Use frequency alongside severity; neither metric alone describes total claim cost.",
    "Action":"Monitor frequency and severity separately by product and region."
})
for region_name, row in region_metrics.sort_values("Claims_to_Premium_%",ascending=False).head(2).iterrows():
    insights.append({
        "Insight":f"Claims-to-premium review: {region_name}",
        "Evidence":f'Claims-to-premium ratio {row["Claims_to_Premium_%"]:.2f}% across {int(row["Policies"])} policies.',
        "Implication":"This is a screening signal and may reflect mix, exposure or premium differences.",
        "Action":"Review exposure, earned premium availability and adjusted regional results before pricing decisions."
    })
for policy_name, row in policy_metrics.sort_values("Claim_Severity",ascending=False).head(2).iterrows():
    insights.append({
        "Insight":f"Severity review: {policy_name}",
        "Evidence":f'Average claim severity {row["Claim_Severity"]:,.2f}; claiming policies {int(row["Claiming_Policies"])}.',
        "Implication":"High average severity can influence aggregate costs even when frequency is lower.",
        "Action":"Review large-claim drivers and coverage distribution for this policy type."
    })
insights.append({
    "Insight":"Random Forest model performance",
    "Evidence":f"Test MAE {mae:,.2f}, RMSE {rmse:,.2f}, R² {r2:.3f}.",
    "Implication":"The model's errors should be considered in the context of the claim amount distribution and baseline.",
    "Action":"Review segment-level errors and validate on a later policy period before operational use."
})
insights_df = pd.DataFrame(insights[:7])
display(insights_df)

## 21. Save model, metrics, cleaned data and reports

In [ ]:
import joblib
from pathlib import Path

OUT = Path("/content/insureiq_outputs")
OUT.mkdir(parents=True,exist_ok=True)

joblib.dump(best_model, OUT/"random_forest_claim_amount_pipeline.joblib")
df.to_csv(OUT/"cleaned_insurance_claims.csv",index=False)
metrics.to_csv(OUT/"model_evaluation_metrics.csv",index=False)
comparison.to_csv(OUT/"baseline_comparison.csv",index=False)
region_metrics.to_csv(OUT/"regional_metrics.csv")
policy_metrics.to_csv(OUT/"policy_metrics.csv")
anomalies.to_csv(OUT/"anomaly_review.csv",index=False)
insights_df.to_csv(OUT/"business_insights.csv",index=False)
results.to_csv(OUT/"test_predictions_and_residuals.csv",index=False)
region_eval.to_csv(OUT/"model_performance_by_region.csv")
policy_eval.to_csv(OUT/"model_performance_by_policy_type.csv")

metadata = {
    "model":"RandomForestRegressor",
    "target":target,
    "features":feature_cols,
    "random_state":RANDOM_STATE,
    "best_params":search.best_params_,
    "test_metrics":{"MAE":float(mae),"RMSE":float(rmse),"R2":float(r2),
                    "bias":float(bias),"positive_claim_MAE":float(positive_mae) if np.isfinite(positive_mae) else None,
                    "zero_claim_MAE":float(zero_mae) if np.isfinite(zero_mae) else None}
}
with open(OUT/"model_metadata.json","w") as f:
    json.dump(metadata,f,indent=2)

print("Saved outputs to:", OUT)
print("\nFiles:")
for p in sorted(OUT.iterdir()):
    print(p.name, f"({p.stat().st_size:,} bytes)")

## 22. Download selected outputs

In [ ]:
# Download the trained model
from google.colab import files
files.download("/content/insureiq_outputs/random_forest_claim_amount_pipeline.joblib")
# To download another file, change the path, e.g.:
# files.download("/content/insureiq_outputs/cleaned_insurance_claims.csv")
# To download all results as one ZIP:
import shutil
shutil.make_archive("/content/insureiq_outputs", "zip", "/content/insureiq_outputs")
files.download("/content/insureiq_outputs.zip")

## Interpretation and limitations
- Claim frequency is the share of policies with a positive claim in this dataset's observation window. It is not an exposure-adjusted annual frequency unless policy exposure time is available.
- Claim severity is total incurred claim amount divided by the number of policies with positive claims.
- The claims-to-premium ratio is simplified because earned premium is not included.
- Random Forest regression predicts the numerical claim amount; classification accuracy is not applicable.
- Zero-claim policies are included in training. Report both overall and positive-claim errors because the target is zero-inflated.
- Outliers and anomaly flags are prompts for review, not evidence of fraud.
- Feature importance is predictive, not causal.
- The adjusted regional analysis is descriptive and may be affected by omitted variables, data quality and model specification.
- Synthetic data can demonstrate a workflow but cannot establish real-world pricing adequacy or operational model performance.